In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.xgboost
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score, recall_score
)
from xgboost import XGBClassifier

In [2]:
config = {
    "run_name": "xgb-kfold-binary-14",
    "n_splits": 2,
    "n_estimators": 150,
    "learning_rate": 0.001,
    "max_depth": 35,
    "random_state": 42,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "scale_pos_weight": 6.4,
    "experiment_name": "loan-risk-xgboost",
    "dataset_path": "/mnt/object/train/train_transformed.csv",
    "label_col": "risk_level"
}


In [3]:
# 3. Load dataset
df = pd.read_csv(config["dataset_path"])
X = df.drop(columns=[config["label_col"]])
y = df[config["label_col"]]  # Already binary-mapped in transform step


In [4]:
output_dir = os.path.join("models", config["run_name"])
os.makedirs(output_dir, exist_ok=True)


In [5]:
# 4. Set MLflow experiment
mlflow.set_experiment(config["experiment_name"])

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1746921146045, experiment_id='1', last_update_time=1746921146045, lifecycle_stage='active', name='loan-risk-xgboost', tags={}>

In [45]:
with mlflow.start_run(run_name=config["run_name"]):
    mlflow.log_params({
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "max_depth": config["max_depth"],
        "n_splits": config["n_splits"],
        "scale_pos_weight": config["scale_pos_weight"]
    })

    kf = StratifiedKFold(n_splits=config["n_splits"], shuffle=True, random_state=config["random_state"])
    fold_accuracies = []
    fold_confidences = {0: [], 1: []}
    all_f1_macro = []
    all_f1_weighted = []
    all_preds = []
    all_probs = []
    all_true = []
    all_precision_macro = []
    all_recall_macro = []

    for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        model = XGBClassifier(
            n_estimators=config["n_estimators"],
            learning_rate=config["learning_rate"],
            max_depth=config["max_depth"],
            random_state=config["random_state"],
            objective=config["objective"],
            eval_metric=config["eval_metric"],
            scale_pos_weight=config["scale_pos_weight"]
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        y_prob = model.predict_proba(X_val)[:, 1]

        all_preds.extend(y_pred)
        all_probs.extend(y_prob)
        all_true.extend(y_val)

        for pred_label, prob in zip(y_pred, y_prob):
            fold_confidences[pred_label].append(prob)

        acc = accuracy_score(y_val, y_pred)
        fold_accuracies.append(acc)
        print(f"Fold {fold+1} Accuracy: {acc:.4f}")
        print(classification_report(y_val, y_pred, target_names=["Low (0)", "High (1)"]))

        f1_macro = f1_score(y_val, y_pred, average="macro", zero_division=0)
        f1_weighted = f1_score(y_val, y_pred, average="weighted", zero_division=0)
        precision_macro = precision_score(y_val, y_pred, average="macro", zero_division=0)
        recall_macro = recall_score(y_val, y_pred, average="macro", zero_division=0)

        all_f1_macro.append(f1_macro)
        all_f1_weighted.append(f1_weighted)
        all_precision_macro.append(precision_macro)
        all_recall_macro.append(recall_macro)

    # -------------------------------
    # Overall metrics
    # -------------------------------
    avg_acc = np.mean(fold_accuracies)
    mlflow.log_metric("avg_kfold_accuracy", avg_acc)
    mlflow.log_metric("avg_f1_macro", np.mean(all_f1_macro))
    mlflow.log_metric("avg_f1_weighted", np.mean(all_f1_weighted))
    mlflow.log_metric("avg_precision_macro", np.mean(all_precision_macro))
    mlflow.log_metric("avg_recall_macro", np.mean(all_recall_macro))

    for class_label in [0, 1]:
        if fold_confidences[class_label]:
            avg_conf = np.mean(fold_confidences[class_label])
            mlflow.log_metric(f"avg_confidence_class_{class_label}", avg_conf)
            print(f"Avg confidence for class {class_label}: {avg_conf:.4f}")

    roc_auc = roc_auc_score(all_true, all_probs)
    mlflow.log_metric("roc_auc_overall", roc_auc)
    print(f"ROC-AUC Overall: {roc_auc:.4f}")

    # -------------------------------
    # Confusion Matrix
    # -------------------------------
    cm = confusion_matrix(all_true, all_preds, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low", "High"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title("Confusion Matrix (Overall)")
    cm_path = os.path.join(output_dir, "confusion_matrix.png")
    plt.savefig(cm_path)
    plt.close()
    mlflow.log_artifact(cm_path)

    # -------------------------------
    # Feature Importance
    # -------------------------------
    importances = model.feature_importances_
    feat_imp_df = pd.DataFrame({
        "feature": X.columns,
        "importance": importances
    }).sort_values(by="importance", ascending=False)

    feat_imp_csv = os.path.join(output_dir, "feature_importance.csv")
    feat_imp_df.to_csv(feat_imp_csv, index=False)
    mlflow.log_artifact(feat_imp_csv)

    plot_path = os.path.join(output_dir, "feature_importance_top15.png")
    plt.figure(figsize=(10, 6))
    feat_imp_df.head(15).plot(kind='barh', x='feature', y='importance', legend=False)
    plt.title("Top 15 Feature Importances (XGBoost)")
    plt.tight_layout()
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # -------------------------------
    # Save model
    # -------------------------------
    model_path = os.path.join(output_dir, "model.pth")
    joblib.dump(model, model_path)
    mlflow.log_artifact(model_path)

    print(f"✅ All artifacts saved to {output_dir}")
    print(f"📊 KFold avg accuracy: {avg_acc:.4f}")

Fold 1 Accuracy: 0.7420
              precision    recall  f1-score   support

     Low (0)       0.90      0.79      0.84    782027
    High (1)       0.24      0.41      0.30    122240

    accuracy                           0.74    904267
   macro avg       0.57      0.60      0.57    904267
weighted avg       0.81      0.74      0.77    904267

Fold 2 Accuracy: 0.7399
              precision    recall  f1-score   support

     Low (0)       0.90      0.79      0.84    782027
    High (1)       0.23      0.41      0.30    122240

    accuracy                           0.74    904267
   macro avg       0.56      0.60      0.57    904267
weighted avg       0.81      0.74      0.77    904267

Avg confidence for class 0: 0.4490
Avg confidence for class 1: 0.5280
ROC-AUC Overall: 0.6711
✅ All artifacts saved to models/xgb-kfold-binary-10
📊 KFold avg accuracy: 0.7410
🏃 View run xgb-kfold-binary-10 at: http://129.114.25.120:8000/#/experiments/1/runs/4cac218f869b4642a8df5fd3efa8c841
🧪 View 

<Figure size 1000x600 with 0 Axes>